# Lab Work - 6.6

**Dataset:** 10 samples, 2 features? (actually 3: Outlook, Humidity, Wind), binary class (Play Tennis: Yes/No)

ID | Outlook | Humidity | Wind | Play
---|---------|----------|------|------
01 | Sunny | High | Weak | No
02 | Sunny | High | Strong | No
03 | Overcast | High | Weak | Yes
04 | Rain | High | Weak | Yes
05 | Rain | Normal | Weak | Yes
06 | Rain | Normal | Strong | No
07 | Overcast | Normal | Strong | Yes
08 | Sunny | High | Weak | No
09 | Sunny | Normal | Weak | Yes
10 | Rain | High | Strong | No

Class distribution: Yes=5, No=5. Root node entropy = 1.0 bit (perfectly mixed).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

# Create the dataset
data = {
    'ID': ['01','02','03','04','05','06','07','08','09','10'],
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast','Sunny','Sunny','Rain'],
    'Humidity': ['High','High','High','High','Normal','Normal','Normal','High','Normal','High'],
    'Wind': ['Weak','Strong','Weak','Weak','Weak','Strong','Strong','Weak','Weak','Strong'],
    'Play': ['No','No','Yes','Yes','Yes','No','Yes','No','Yes','No']
}

df = pd.DataFrame(data)
print(df)
print('\nClass distribution:')
print(df['Play'].value_counts())

## Q1: Gini Impurity

In [ ]:
# 01 Gini Impurity Formula
print("Gini(t) = 1 - Σ_k p_k² where p_k is fraction of class k in node t")
print("Pure node (all one class): Gini=0")
print("Maximally impure binary node: Gini=0.5")

In [ ]:
# 02 Root Gini
p_yes = 5/10
p_no = 5/10
gini_root = 1 - (p_yes**2 + p_no**2)
print(f"Root Gini: {gini_root}")

In [ ]:
# Helper function for Gini
def gini_impurity(y):
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return 1 - np.sum(probs**2)

print("Root Gini (function):", gini_impurity(df['Play']))

In [ ]:
# 03 Outlook Split
print("Outlook groups:")
sunny = df[df['Outlook']=='Sunny']
overcast = df[df['Outlook']=='Overcast']
rain = df[df['Outlook']=='Rain']

print("Sunny:", sunny['Play'].value_counts().to_dict())
print("Overcast:", overcast['Play'].value_counts().to_dict())
print("Rain:", rain['Play'].value_counts().to_dict())

gini_sunny = gini_impurity(sunny['Play'])
gini_overcast = gini_impurity(overcast['Play'])
gini_rain = gini_impurity(rain['Play'])

print(f"Gini Sunny: {gini_sunny}")
print(f"Gini Overcast: {gini_overcast}")
print(f"Gini Rain: {gini_rain}")

In [ ]:
# 04 Weighted Gini after Outlook split
n_sunny = len(sunny)
n_overcast = len(overcast)
n_rain = len(rain)
n_total = len(df)

weighted_gini_outlook = (n_sunny/n_total)*gini_sunny + (n_overcast/n_total)*gini_overcast + (n_rain/n_total)*gini_rain
print(f"Weighted Gini after Outlook split: {weighted_gini_outlook}")

In [ ]:
# 05 Gini Gain for Outlook
gini_gain_outlook = gini_root - weighted_gini_outlook
print(f"Gini Gain (Outlook): {gini_gain_outlook}")

In [ ]:
# 06 Humidity & Wind splits + comparison
print("\n=== Humidity Split (High vs Normal) ===")
high = df[df['Humidity']=='High']
normal = df[df['Humidity']=='Normal']
gini_high = gini_impurity(high['Play'])
gini_normal = gini_impurity(normal['Play'])
weighted_gini_hum = (len(high)/n_total)*gini_high + (len(normal)/n_total)*gini_normal
gini_gain_hum = gini_root - weighted_gini_hum
print(f"Gini Gain (Humidity): {gini_gain_hum}")

print("\n=== Wind Split (Weak vs Strong) ===")
weak = df[df['Wind']=='Weak']
strong = df[df['Wind']=='Strong']
gini_weak = gini_impurity(weak['Play'])
gini_strong = gini_impurity(strong['Play'])
weighted_gini_wind = (len(weak)/n_total)*gini_weak + (len(strong)/n_total)*gini_strong
gini_gain_wind = gini_root - weighted_gini_wind
print(f"Gini Gain (Wind): {gini_gain_wind}")

print("\nBest split by Gini Gain:")
gains = {'Outlook': gini_gain_outlook, 'Humidity': gini_gain_hum, 'Wind': gini_gain_wind}
best_gini = max(gains, key=gains.get)
print(gains)
print("Best feature:", best_gini)

## Q2: Entropy & Information Gain

In [ ]:
# 01 Shannon Entropy
print("H(t) = - Σ_k p_k log₂(p_k)")
print("Pure node: H=0")
print("Balanced binary: H=1")

In [ ]:
# 02 Root Entropy
def entropy(y):
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return -np.sum(probs * np.log2(probs))

h_root = entropy(df['Play'])
print(f"Root Entropy: {h_root}")

In [ ]:
# 03 Entropy for Outlook children
print("Sunny child entropy:", entropy(sunny['Play']))
print("Overcast child entropy:", entropy(overcast['Play']))
print("Rain child entropy:", entropy(rain['Play']))

In [ ]:
# 04 Information Gain for Outlook
ig_outlook = h_root - ((len(sunny)/n_total)*entropy(sunny['Play']) + 
                       (len(overcast)/n_total)*entropy(overcast['Play']) + 
                       (len(rain)/n_total)*entropy(rain['Play']))
print(f"Information Gain (Outlook): {ig_outlook}")

In [ ]:
# 05 IG for Humidity and Wind
ig_hum = h_root - ((len(high)/n_total)*entropy(high['Play']) + (len(normal)/n_total)*entropy(normal['Play']))
ig_wind = h_root - ((len(weak)/n_total)*entropy(weak['Play']) + (len(strong)/n_total)*entropy(strong['Play']))

print(f"IG Humidity: {ig_hum}")
print(f"IG Wind: {ig_wind}")

print("\nComparison Table:")
comparison = pd.DataFrame({
    'Feature': ['Outlook', 'Humidity', 'Wind'],
    'Gini Gain': [gini_gain_outlook, gini_gain_hum, gini_gain_wind],
    'Information Gain': [ig_outlook, ig_hum, ig_wind]
})
print(comparison)
print("\nBest split same for both criteria?", best_gini == 'Outlook')

## Q3: Build the Full Tree

In [ ]:
# Using best root split: Outlook
print("Root split: Outlook")
print("Sunny branch samples:", sunny['ID'].tolist())
print("Overcast branch samples:", overcast['ID'].tolist())
print("Rain branch samples:", rain['ID'].tolist())

In [ ]:
# Overcast is pure -> Leaf: Yes
print("Overcast node: Pure Yes leaf (Gini=0)")

In [ ]:
# Continue with Sunny and Rain branches (manual split finding)
print("\nSunny branch (IDs: 01,02,08,09):")
print(sunny[['ID','Humidity','Wind','Play']])

In [ ]:
# Best split in Sunny: Humidity (High vs Normal)
sunny_high = sunny[sunny['Humidity']=='High']  # all No
sunny_normal = sunny[sunny['Humidity']=='Normal']  # all Yes
print("Sunny High: all No")
print("Sunny Normal: all Yes")

In [ ]:
print("\nRain branch (IDs: 04,05,06,10):")
print(rain[['ID','Humidity','Wind','Play']])

In [ ]:
# Best split in Rain: Wind (Weak vs Strong)
rain_weak = rain[rain['Wind']=='Weak']   # all Yes
rain_strong = rain[rain['Wind']=='Strong'] # all No
print("Rain Weak: all Yes")
print("Rain Strong: all No")

In [ ]:
# Tree Structure Summary
print("""
Full Decision Tree:
Root: Outlook?
├── Overcast → Yes (leaf)
├── Sunny
│   ├── Humidity=High → No (leaf)
│   └── Humidity=Normal → Yes (leaf)
└── Rain
    ├── Wind=Weak → Yes (leaf)
    └── Wind=Strong → No (leaf)
""")

In [ ]:
# Visualize with sklearn (for reference)
X = pd.get_dummies(df[['Outlook','Humidity','Wind']])
y = df['Play'].map({'Yes':1, 'No':0})

tree = DecisionTreeClassifier(criterion='gini', max_depth=None)
tree.fit(X, y)

plt.figure(figsize=(12,8))
plot_tree(tree, feature_names=X.columns, class_names=['No','Yes'], filled=True)
plt.title("Decision Tree (sklearn)")
plt.show()

## Q4: Decision Boundary & Theory

**Decision Regions:** Axis-aligned rectangles created by successive splits on individual features.

Decision Trees **cannot** represent diagonal boundaries directly. They approximate them with many axis-aligned splits (staircase effect).

**Gini vs Entropy:**
- Both measure impurity.
- scikit-learn default: **Gini** (faster).
- Entropy (Information Gain) tends to produce slightly more balanced trees.

**Overfitting:** A full tree achieves 100% training accuracy because it can create a leaf for every training sample.

Hyperparameters to control overfitting:
1. `max_depth` → reduce it
2. `min_samples_split` → increase it
3. `min_samples_leaf` → increase it

In [ ]:
print("Lab complete! Tree built manually and with sklearn.")